In [ ]:
from pathlib import Path
import sys
import os
from dotenv import load_dotenv
import cv2
import importlib

sys.path.append(str(Path.cwd().parent / "src"))
load_dotenv()

p = Path.cwd().parent / "datasets" / "s3" / "input"
p.mkdir(parents=True, exist_ok=True)

In [ ]:
from commons.s3 import S3Client

# from ecg_scanner.scanner import ECGScanner
from utils import plot_stages

In [ ]:
s3_client = S3Client(os.getenv("S3_BUCKET_NAME"))
files = [f for f in s3_client.list_files() if f.endswith(".png")]
images = []
for f in files:
    file_name = p / f.split("/")[-1]
    if file_name.is_file():
        images.append(cv2.imread(file_name))
        continue
    s3_client.download_file(f, file_name)
    images.append(cv2.imread(file_name))

In [ ]:
import ecg_scanner.scanner as ecg_scanner

importlib.reload(ecg_scanner)

config = ecg_scanner.ECGScannerConfig(
    debug_mode=True,
    hsv_hue_multiplier=0.5,
    hsv_saturation_multiplier=1.8,
    hsv_brightness_multiplier=0.4,
    bilateral_filter_d=7,
    bilateral_filter_sigma_color=25,
    bilateral_filter_sigma_space=25,
)

scanner = ecg_scanner.ECGScanner(config=config)
for i, image in enumerate(images):
    if i == 15:
        break
    scanner.scan(image)
    plot_stages(scanner.get_stages())
    scanner.reset_stages()